In [1]:
import numpy as np
from scipy.special import eval_hermitenorm

rng = np.random.default_rng(42)

Transformer une variable gaussienne en variable hermitienne

Juste pour essayer, on s'arrête à 3, juste pour voir

In [2]:
# Variable gaussienne standard
Z = rng.standard_normal(10)

# Transformations de Hermite
H1 = eval_hermitenorm(1, Z)   # = Z
H2 = eval_hermitenorm(2, Z)   # = Z**2 - 1
H3 = eval_hermitenorm(3, Z)   # = Z**3 - 3*Z

print("Z  =", Z)
print("H2 =", H2)
print("H3 =", H3)

Z  = [ 0.30471708 -1.03998411  0.7504512   0.94056472 -1.95103519 -1.30217951
  0.1278404  -0.31624259 -0.01680116 -0.85304393]
H2 = [-0.9071475   0.08156694 -0.436823   -0.11533801  2.80653831  0.69567147
 -0.98365683 -0.89999062 -0.99971772 -0.27231606]
H3 = [-0.8858575   1.99513989 -1.82871674 -1.9896123  -1.57358462  1.69846988
 -0.38143189  0.91710055  0.05039873  1.93838541]


bon mais maintenant, on fait le lien avec la gaussienne

In [3]:
T = 1.0
N = 252
dt = T / N

# Incréments browniens
Z = rng.standard_normal(N)
dW = np.sqrt(dt) * Z

# Variable d'Hermite associée aux incréments
# On remet l'incrément sous forme standard normale
X2 = eval_hermitenorm(2, dW / np.sqrt(dt))   # = Z**2 - 1
X3 = eval_hermitenorm(3, dW / np.sqrt(dt))   # = Z**3 - 3Z

sauf que là on n'a juste fait des évaluations, on n'a pas trouvé la base associée.

In [5]:
import numpy as np
from math import factorial

In [6]:
def hermite_probabilists_basis(x, degree):
    x = np.asarray(x, dtype=float).reshape(-1)
    n_samples = x.shape[0]

    H = np.zeros((n_samples, degree + 1), dtype=float)
    H[:, 0] = 1.0

    if degree >= 1:
        H[:, 1] = x

    for n in range(1, degree):
        H[:, n + 1] = x * H[:, n] - n * H[:, n - 1]

    return H


def project_on_hermite(y, g, degree, mu=None, sigma=None, standardize=True):
    """
    Projette empiriquement Y sur les polynômes d'Hermite jusqu'au degré.
    - si standardize=True, on suppose que g est gaussienne (éventuellement N(mu, sigma^2)) et on la standardise en z = (g - mu) / sigma ;
    - la projection est alors faite sur la base H_n(z), orthogonale pour N(0,1).

    Paramètres
    y : array-like, shape (n_samples,)
        Réalisations de la variable à projeter, typiquement y = f(g).
    g : array-like, shape (n_samples,)
        Réalisations de la variable gaussienne.
    degree : int
        Degré maximal de la projection de Hermite.
    mu : float ou None
        Moyenne de g. Si None et standardize=True, estimée empiriquement.
    sigma : float ou None
        Écart-type de g. Si None et standardize=True, estimé empiriquement.
    standardize : bool
        Si True, remplace g par z = (g - mu) / sigma.

    Retourne:
    result : dict
        - "coeffs"      : coefficients a_0,...,a_degree
        - "projection"  : valeurs projetées sur l'échantillon fourni
        - "basis"       : matrice des H_n(z)
        - "z"           : variable gaussienne standardisée utilisée
        - "mu"          : moyenne utilisée
        - "sigma"       : écart-type utilisé
    """
    y = np.asarray(y, dtype=float).reshape(-1)
    g = np.asarray(g, dtype=float).reshape(-1)

    if y.shape[0] != g.shape[0]:
        raise ValueError("`y` et `g` doivent avoir la même taille.")

    if degree < 0:
        raise ValueError("`degree` doit être >= 0.")

    if standardize:
        if mu is None:
            mu = np.mean(g)
        if sigma is None:
            sigma = np.std(g, ddof=0)

        if sigma <= 0:
            raise ValueError("`sigma` doit être strictement positif.")

        z = (g - mu) / sigma
    else:
        z = g
        mu = 0.0 if mu is None else mu
        sigma = 1.0 if sigma is None else sigma

    H = hermite_probabilists_basis(z, degree)

    # Projection orthogonale empirique :
    # a_n ≈ E[Y H_n(Z)] / n!  ≈ mean(Y H_n(Z)) / n!
    coeffs = np.array([
        np.mean(y * H[:, n]) / factorial(n)
        for n in range(degree + 1)
    ])

    projection = H @ coeffs

    return {
        "coeffs": coeffs,
        "projection": projection,
        "basis": H,
        "z": z,
        "mu": mu,
        "sigma": sigma
    }


def evaluate_hermite_projection(g_new, coeffs, mu=0.0, sigma=1.0, standardize=True):
    """
    Évalue la projection d'Hermite sur de nouveaux points.

    Paramètres
    ----------
    g_new : array-like
        Nouveaux points de la variable gaussienne.
    coeffs : array-like
        Coefficients de la projection.
    mu, sigma : float
        Paramètres de standardisation.
    standardize : bool
        Si True, utilise z = (g_new - mu) / sigma.

    Retour
    ------
    y_hat : ndarray
        Valeurs de la projection aux nouveaux points.
    """
    g_new = np.asarray(g_new, dtype=float).reshape(-1)
    coeffs = np.asarray(coeffs, dtype=float).reshape(-1)

    if standardize:
        if sigma <= 0:
            raise ValueError("`sigma` doit être strictement positif.")
        z_new = (g_new - mu) / sigma
    else:
        z_new = g_new

    H_new = hermite_probabilists_basis(z_new, len(coeffs) - 1)
    return H_new @ coeffs

In [7]:
def check_hermite_orthogonality(z, degree):
    """
    Vérifie empiriquement l'orthogonalité de H_0,...,H_degree
    pour un échantillon z censé suivre N(0,1).
    """
    H = hermite_probabilists_basis(z, degree)

    n_samples = len(z)

    # Matrice de Gram empirique :
    # G[n,m] ≈ E[H_n(Z) H_m(Z)]
    G = (H.T @ H) / n_samples

    # Matrice théorique : diag(n!)
    G_theoretical = np.diag([factorial(n) for n in range(degree + 1)])

    # Erreur
    error = G - G_theoretical

    return {
        "gram_empirical": G,
        "gram_theoretical": G_theoretical,
        "error": error
    }

Sur la diagonale, on doit trouver n! et hors-diagonal, on doit trouver 0. L'erreur doit être proche de 0.

Bon, maintenant, on essaie de faire de la réduction de variance


In [8]:
def hermite_probabilists_basis(x, degree):
    x = np.asarray(x, dtype=float).reshape(-1)
    H = np.zeros((len(x), degree + 1))
    H[:, 0] = 1.0
    if degree >= 1:
        H[:, 1] = x
    for n in range(1, degree):
        H[:, n + 1] = x * H[:, n] - n * H[:, n - 1]
    return H


def fit_hermite_projection(y, z, degree):
    """
    Ajuste la projection de y sur H_0,...,H_degree.
    Ici z doit être ~ N(0,1).
    """
    y = np.asarray(y, dtype=float).reshape(-1)
    z = np.asarray(z, dtype=float).reshape(-1)

    H = hermite_probabilists_basis(z, degree)

    coeffs = np.array([
        np.mean(y * H[:, n]) / factorial(n)
        for n in range(degree + 1)
    ])

    return coeffs


def eval_hermite_projection(z, coeffs):
    z = np.asarray(z, dtype=float).reshape(-1)
    H = hermite_probabilists_basis(z, len(coeffs) - 1)
    return H @ coeffs


def monte_carlo_with_hermite_cv(payoff_func, degree, n_train, n_test, seed=None):
    """
    Réduction de variance par projection d'Hermite.

    Étape 1 : on estime les coefficients sur un échantillon d'entraînement.
    Étape 2 : on applique la variable de contrôle sur un échantillon indépendant.

    Retourne :
    - estimateur MC brut
    - estimateur CV
    - variance empirique brute
    - variance empirique réduite
    - coefficients Hermite
    """
    rng = np.random.default_rng(seed)

    # Échantillon d'entraînement
    z_train = rng.standard_normal(n_train)
    y_train = payoff_func(z_train)

    coeffs = fit_hermite_projection(y_train, z_train, degree)

    # Échantillon de test indépendant
    z_test = rng.standard_normal(n_test)
    y_test = payoff_func(z_test)

    proj_test = eval_hermite_projection(z_test, coeffs)

    # Partie non constante de la projection = variable de contrôle de moyenne nulle
    control = proj_test - coeffs[0]

    # estimateur MC brut
    mc_estimator = np.mean(y_test)

    # estimateur CV, beta=1
    cv_samples = y_test - control
    cv_estimator = np.mean(cv_samples)

    # variances empiriques des variables échantillonnées
    raw_var = np.var(y_test, ddof=1)
    cv_var = np.var(cv_samples, ddof=1)

    return {
        "mc_estimator": mc_estimator,
        "cv_estimator": cv_estimator,
        "raw_variance": raw_var,
        "cv_variance": cv_var,
        "variance_reduction_factor": raw_var / cv_var if cv_var > 0 else np.inf,
        "coeffs": coeffs
    }

In [17]:
def payoff(z):
    return np.exp(0.5 * z)

res = monte_carlo_with_hermite_cv(
    payoff_func=payoff,
    degree=4,
    n_train=50_000,
    n_test=50_000,
    seed=123
)

print("Estimateur MC brut     :", res["mc_estimator"])
print("Estimateur CV Hermite  :", res["cv_estimator"])
print("Variance brute         :", res["raw_variance"])
print("Variance réduite       :", res["cv_variance"])
print("Facteur de réduction   :", res["variance_reduction_factor"])
print("Coefficients           :", res["coeffs"])

Estimateur MC brut     : 1.133159755331047
Estimateur CV Hermite  : 1.1331930713284868
Variance brute         : 0.3666030612418007
Variance réduite       : 0.00010076584005117741
Facteur de réduction   : 3638.168064252813
Coefficients           : [1.13464857 0.56998315 0.14191029 0.02182462 0.00137702]


Maintenant on applique avec Monte Carlo

In [10]:
def monte_carlo_brownian_increment(payoff_func, dt, degree, n_train, n_test, seed=None):
    rng = np.random.default_rng(seed)

    dW_train = np.sqrt(dt) * rng.standard_normal(n_train)
    z_train = dW_train / np.sqrt(dt)
    y_train = payoff_func(dW_train)

    coeffs = fit_hermite_projection(y_train, z_train, degree)

    dW_test = np.sqrt(dt) * rng.standard_normal(n_test)
    z_test = dW_test / np.sqrt(dt)
    y_test = payoff_func(dW_test)

    proj_test = eval_hermite_projection(z_test, coeffs)
    control = proj_test - coeffs[0]

    mc_estimator = np.mean(y_test)
    cv_samples = y_test - control
    cv_estimator = np.mean(cv_samples)

    return {
        "mc_estimator": mc_estimator,
        "cv_estimator": cv_estimator,
        "raw_variance": np.var(y_test, ddof=1),
        "cv_variance": np.var(cv_samples, ddof=1),
        "coeffs": coeffs
    }